# G1 — The shared space as an object: one hub, five encoders, zero-shot
# component transfer

Every result so far is *pairwise*: encoder A relates to encoder B by a
linear map. N encoders would need N-squared maps, and each map is fitted
against its own target, so nothing yet shows that the encoders share
**one** coordinate system rather than a web of bilateral agreements.

This notebook tests the stronger claim. A single hub space is
constructed — a whitened PCA basis, fitted on training rows of one
reference encoder — and every encoder is given one linear map into it.
Two questions follow, and the second is the substantive one.

**1. Transitivity.** Fit A→hub and hub→B, never fitting A→B. Does the
composition match a directly fitted A→B? If the hub were an arbitrary
intermediate, composing two lossy maps would degrade badly. If the
encoders genuinely share a coordinate system, the composition should
approach the direct fit.

**2. Zero-shot component transfer.** Train a caption-prediction head on
hub vectors from **one** encoder only. Then feed a **different**
encoder's data through its own hub map into the same head. The head has
never seen that encoder, at any point, in any form. If retrieval still
works, a component built in the hub is portable across encoders that
were never trained together — which is the shared space demonstrated by
use rather than by correlation.

**Pre-registered reading.** Composition within 15 per cent of the direct
fit counts as transitivity holding. Transfer retaining at least half the
native head's R@1 counts as portability; below a quarter counts as
failure. A random-map control must sit at chance, or the measurement is
meaningless.

**Data: nothing new is encoded.** The DINOv2 sweep used nested prefixes
of the same sorted COCO id list, so the three image caches share their
first 9,533 rows, and the E1/E1.1 pairs files were built over the same
ordering. Alignment is verified numerically below rather than
assumed.

In [ ]:
import os
from pathlib import Path
STORAGE   = "drive"                     # "drive" | "local" | "env"
DRIVE_DIR = "/content/drive/MyDrive/convergence_experiment"
LOCAL_DIR = "./convergence_data"
try:
    import google.colab                 # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
if STORAGE == "env":
    assert os.environ.get("DATA_DIR"), "STORAGE='env' but DATA_DIR is unset"
elif STORAGE == "drive" and IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    os.environ["DATA_DIR"] = DRIVE_DIR
else:
    if STORAGE == "drive":
        print("not on Colab - using LOCAL_DIR")
    os.environ["DATA_DIR"] = str(Path(LOCAL_DIR).resolve())
DATA_DIR = Path(os.environ["DATA_DIR"])
print("DATA_DIR:", DATA_DIR)

In [ ]:
import numpy as np

# ---- gather every cached space, and VERIFY they are row-aligned ----
SPACES, N_COMMON = {}, None
for size in ("small", "base", "large"):
    f = DATA_DIR / f"e1_img_ckpt_dinov2-{size}_cls+patch.npz"
    if f.exists():
        d = np.load(str(f))
        SPACES[f"img_{size}"] = d["img"].astype(np.float64)
        n = len(d["img"])
        N_COMMON = n if N_COMMON is None else min(N_COMMON, n)
        print(f"  img_{size:5s} {d['img'].shape}")
assert len(SPACES) >= 2, "need at least two image caches - run E1 first"

def add_text(candidates, key):
    """Attach a text space from whichever cached pairs file is present.

    The pairs files carry no ids, so alignment is PROVEN rather than
    assumed: the file's image block must match one of the image caches
    already loaded. Several filenames are tried because early runs were
    saved before the encoder size was added to the name - and for a text
    encoder the image side is irrelevant anyway, since captions are
    encoded independently of it."""
    for fname in candidates:
        f = DATA_DIR / fname
        if not f.exists():
            continue
        d = np.load(str(f))
        for ref in [k for k in SPACES if k.startswith("img_")]:
            # width must match before values can be compared at all
            if d["img"].shape[1] != SPACES[ref].shape[1]:
                continue
            n = min(len(d["img"]), len(SPACES[ref]))
            if n < 500:
                continue
            if np.allclose(d["img"][:n], SPACES[ref][:n], atol=1e-4):
                SPACES[key] = d["txt"].astype(np.float64)[:n]
                print(f"  {key:11s} {SPACES[key].shape}  from {fname} "
                      f"(alignment verified against {ref})")
                return
        print(f"  ({fname} present but matches no image cache - skipped)")
    print(f"  (no usable file for {key}: tried {', '.join(candidates)})")

add_text(["crossmodal_pairs.npz"], "txt_bge")
add_text(["crossmodal_pairs_gpt2.npz",                       # pre-rename
          "crossmodal_pairs_gpt2_dinov2-base_cls+patch.npz",
          "crossmodal_pairs_gpt2_dinov2-large_cls+patch.npz",
          "crossmodal_pairs_gpt2_dinov2-small_cls+patch.npz"], "txt_gpt2")

# E1.3's ladder encoders - four training objectives in one hub.
# Skipped automatically if E1.3 has not been run. These files store the
# text array under "txt" and carry no image block, so alignment is by
# construction (same rows, same id list) rather than provable here.
for _tag, _fn in (("txt_bert", "e13_txt_bert.npz"),
                  ("txt_sbert", "e13_txt_sbert.npz")):
    _f = DATA_DIR / _fn
    if _f.exists():
        _d = np.load(str(_f))
        _key = "txt" if "txt" in _d.files else _d.files[0]
        SPACES[_tag] = _d[_key].astype(np.float64)
        print(f"  {_tag:11s} {SPACES[_tag].shape}  from {_fn}")
    else:
        print(f"  (no {_fn} - run E1.3 to add {_tag})")

N_COMMON = min(min(len(v) for v in SPACES.values()), N_COMMON)
SPACES = {k: v[:N_COMMON] for k, v in SPACES.items()}
print(f"\n{len(SPACES)} spaces on {N_COMMON} common items: "
      f"{list(SPACES)}")

rng = np.random.default_rng(0)
perm = rng.permutation(N_COMMON)
N_EVAL = 1000
te, tr = perm[:N_EVAL], perm[N_EVAL:]
print(f"{len(tr)} train / {len(te)} eval")
for k, v in SPACES.items():
    print(f"   {k:11s} width {v.shape[1]:5d}  "
          f"{len(tr)/v.shape[1]:5.1f} rows per input dim")

## Building the hub

The hub is a whitened PCA basis of one reference encoder, estimated on
training rows only. Whitening is used because Section C.11 showed that
an isotropic target is what makes cosine readable — the hub is the space
components will be *read in*, so it should be isotropic. (Section C.12
showed the opposite prescription applies to *discovering* an unknown
correspondence; that is not what happens here, since every map is fitted
on known rows.)

## Shape agreement before the hub exists

A one-cell check that needs no map: does each pair of spaces already
agree on how far apart the same items are? Where it is high, the hub is
supplying coordinates for a correspondence that already exists; where it
is near zero, no hub can manufacture one. Run it before reading the
transfer numbers.

In [ ]:
from scipy.stats import spearmanr

# self-contained: this cell runs before the hub cell defines its helpers,
# so it brings its own. Nothing here is fitted.
def _l2(V):
    return V / (np.linalg.norm(V, axis=-1, keepdims=True) + 1e-12)

_names = list(SPACES)
SHAPE_N = min(400, len(te))
_si = rng.permutation(len(te))[:SHAPE_N]
_iu = np.triu_indices(SHAPE_N, 1)
def _pd(Z):
    Zn = _l2(Z); return (1.0 - Zn @ Zn.T)[_iu]
_D = {k: _pd(SPACES[k][te][_si]) for k in _names}
print(f"shape agreement, {SHAPE_N} held-out items, no fitted map\n")
print(f"{'pair':26s} {'dist rho':>9s}")
for a in _names:
    for b in _names:
        if a >= b: continue
        print(f"{a+' <-> '+b:26s} "
              f"{float(spearmanr(_D[a], _D[b]).statistic):+9.3f}")
print("\nhigh here = a correspondence already exists and the hub only")
print("has to supply coordinates for it (report C.11.2).")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mp
from scipy.stats import spearmanr

# ---- every pair's shape agreement, grouped by what the pair SPANS ----
# Cross-modal pairs are the interesting ones: a vision encoder that never
# saw text and a text encoder that never saw an image have no objective
# on either side rewarding agreement, so whatever they share was not put
# there by training either of them to agree.
_RHO = {}
for a in _names:
    for b in _names:
        if a >= b: continue
        _RHO[(a, b)] = float(spearmanr(_D[a], _D[b]).statistic)

def _kind(a, b):
    ia, ib = a.startswith("img"), b.startswith("img")
    return ("image-image" if ia and ib else
            "text-text" if not ia and not ib else "IMAGE-TEXT")
_COL = {"image-image": "#1a5276", "text-text": "#0f766e",
        "IMAGE-TEXT": "#b45309"}

def plot_shape(pair=None):
    """Panel A: every pair ranked. Panel B: grouped by span.
    Panel C: the strongest CROSS-MODAL pair, drawn."""
    fig = plt.figure(figsize=(13.4, 5.4))
    gs = fig.add_gridspec(1, 3, width_ratios=[1.45, 0.9, 1.05], wspace=0.34)

    ax = fig.add_subplot(gs[0])
    items = sorted(_RHO.items(), key=lambda kv: kv[1])
    ax.barh(range(len(items)), [v for _, v in items],
            color=[_COL[_kind(*k)] for k, _ in items])
    ax.set_yticks(range(len(items)))
    ax.set_yticklabels([f"{a} - {b}" for (a, b), _ in items], fontsize=6.2)
    ax.set_xlabel("distance-rank correlation (Spearman rho)", fontsize=8.2)
    ax.set_title("A · every pair, no fitted map", fontsize=9.8,
                 color="#1a1a2e")
    ax.grid(alpha=0.2, axis="x"); ax.tick_params(labelsize=7)
    ax.legend(handles=[mp.Patch(facecolor=c, label=k)
                       for k, c in _COL.items()],
              fontsize=7, frameon=False, loc="lower right")

    ax = fig.add_subplot(gs[1])
    groups = {}
    for k, v in _RHO.items():
        groups.setdefault(_kind(*k), []).append(v)
    order = [g for g in ("image-image", "IMAGE-TEXT", "text-text")
             if g in groups]
    means = [float(np.mean(groups[g])) for g in order]
    ax.bar(range(len(order)), means, color=[_COL[g] for g in order],
           width=0.6)
    for i, g in enumerate(order):
        ax.scatter([i]*len(groups[g]), groups[g], s=15, c="white",
                   edgecolors="#1a1a2e", zorder=3, linewidths=0.8)
        ax.text(i, means[i]+0.025, f"{means[i]:.2f}", ha="center",
                fontsize=9.5, color="#1a1a2e", weight="bold")
    ax.set_xticks(range(len(order)))
    ax.set_xticklabels([g.replace("-", "\n") for g in order], fontsize=7.4)
    ax.set_ylabel("Spearman rho", fontsize=8.2)
    ax.set_title("B · grouped by span", fontsize=9.8, color="#1a1a2e")
    ax.grid(alpha=0.2, axis="y"); ax.tick_params(labelsize=7.4)

    # panel C defaults to the strongest CROSS-MODAL pair, not the
    # strongest overall - that is the one worth drawing
    if pair is None:
        cm = {k: v for k, v in _RHO.items() if _kind(*k) == "IMAGE-TEXT"}
        pair = max(cm, key=cm.get) if cm else max(_RHO, key=_RHO.get)
    a, b = pair
    ax = fig.add_subplot(gs[2])
    s = rng.permutation(len(_D[a]))[:6000]
    ax.scatter(_D[a][s], _D[b][s], s=3, alpha=0.18, c=_COL[_kind(a, b)],
               edgecolors="none")
    z = np.polyfit(_D[a][s], _D[b][s], 1)
    xs = np.linspace(_D[a].min(), _D[a].max(), 50)
    ax.plot(xs, np.polyval(z, xs), c="#c0392b", lw=1.6, ls="--")
    ax.set_xlabel(f"cosine distance in {a}", fontsize=8.2)
    ax.set_ylabel(f"cosine distance in {b}", fontsize=8.2)
    ax.set_title(f"C · {a} vs {b}\nrho = {_RHO[pair]:+.3f}",
                 fontsize=9.8, color="#1a1a2e")
    ax.grid(alpha=0.2); ax.tick_params(labelsize=7.2)

    for x in fig.axes:
        x.spines["top"].set_visible(False)
        x.spines["right"].set_visible(False)
    fig.suptitle("Shape agreement before the hub exists - grouped by what "
                 "each pair spans", fontsize=11, color="#1a1a2e")
    # NOT tight_layout(): the gridspec width_ratios and wspace above are
    # deliberate, and tight_layout overrides them (and warns about it).
    fig.subplots_adjust(left=0.13, right=0.97, top=0.83, bottom=0.14,
                        wspace=0.34)
    plt.show()

plot_shape()

# ---- the reading, stated from the measured groups ----
_g = {}
for k, v in _RHO.items():
    _g.setdefault(_kind(*k), []).append(v)
if "IMAGE-TEXT" in _g:
    print(f"cross-modal mean rho: {np.mean(_g['IMAGE-TEXT']):+.3f}")
    print("  a vision encoder that never saw text and a text encoder that")
    print("  never saw an image, agreeing on how far apart the same items")
    print("  are - with no map fitted and no objective on either side")
    print("  rewarding agreement with the other.")
if "image-image" in _g and "text-text" in _g:
    print(f"\nimage-image {np.mean(_g['image-image']):+.3f}   "
          f"text-text {np.mean(_g['text-text']):+.3f}")
    print("  Read these against LINEAGE, not just modality: the DINOv2")
    print("  encoders are distilled from one ViT-g teacher, so part of")
    print("  their agreement is shared ancestry. Any text pair scoring")
    print("  ABOVE them shares an objective but no lineage - which would")
    print("  say a shared training goal produces more shape agreement")
    print("  than shared weights do.")

In [ ]:
# A 256-d hub cannot RECONSTRUCT a 768- or 1024-d target: measured on
# the Experiment B spaces, retention into the widest target went from
# -444% at 256 to +84% at 2048. Reconstruction needs width; the transfer
# test below does not, and passed even at 256. Sweep this against the
# rows-per-hub-dim ratio printed above - more width is not free.
# 512 is the measured operating point, not a guess: the width x alpha
# sweep below found a TRANSFER cliff between 512 and 768 (whitened PCA
# starts amplifying noise-dominated tail directions past it). Wider
# hubs reconstruct better and transfer worse - the two demands are not
# reconcilable, which is itself the finding. Read the transfer column
# first, retention second.
HUB_DIM  = 512
ALPHA    = 1e-2

# HUB_MODE decides what the hub is a basis OF, and it matters:
#   "single"   - whitened PCA of one reference encoder. Cheap, but the
#                hub then represents that encoder's modality well and
#                others poorly, so maps INTO the under-represented
#                spaces lose more than they should.
#   "balanced" - whitened PCA of every space concatenated, each first
#                standardised so no space dominates by width or scale.
#                Costs nothing extra and removes the blind spot.
HUB_MODE = "balanced"            # "balanced" | "single"
HUB_REF  = "img_base" if "img_base" in SPACES else list(SPACES)[0]

def l2n(V):
    return V / (np.linalg.norm(V, axis=-1, keepdims=True) + 1e-12)
def ridge(X, Y, a=ALPHA):
    return np.linalg.solve(X.T @ X + a * np.eye(X.shape[1]), X.T @ Y)
def r2(Y, P):
    return float(1 - ((Y - P) ** 2).sum() / ((Y - Y.mean(0)) ** 2).sum())
def recall(S):
    o = np.argsort(-S, 1); r = (o == np.arange(len(S))[:, None]).argmax(1)
    return {k: float((r < k).mean()) for k in (1, 5, 10)}

if HUB_MODE == "single":
    ref = SPACES[HUB_REF][tr]
else:
    # standardise each space before concatenating, so a wider or
    # higher-variance space cannot dominate the shared basis
    parts = []
    for k in SPACES:
        v = SPACES[k][tr]
        parts.append((v - v.mean(0)) / (v.std(0).mean() + 1e-12))
    ref = np.hstack(parts)
    print(f"balanced hub over {len(parts)} spaces, "
          f"concatenated width {ref.shape[1]}")

mu = ref.mean(0)
_U, _s, _Vt = np.linalg.svd(ref - mu, full_matrices=False)
K = min(HUB_DIM, len(_s))
BASIS = _Vt[:K].T / (_s[:K] / np.sqrt(len(ref)))          # whitened PCA
HUB_TR = (ref - mu) @ BASIS            # hub coordinates of the TRAIN rows

def hub_coords(which):
    """Hub coordinates for train/eval rows, built the same way for both."""
    if HUB_MODE == "single":
        return (SPACES[HUB_REF][which] - mu) @ BASIS
    parts = []
    for k in SPACES:
        v = SPACES[k][which]
        vt = SPACES[k][tr]
        parts.append((v - vt.mean(0)) / (vt.std(0).mean() + 1e-12))
    return (np.hstack(parts) - mu) @ BASIS

HUB = {"tr": HUB_TR, "te": hub_coords(te)}

# one map per encoder INTO the hub - fitted on train rows only
TO_HUB = {k: ridge(v[tr], HUB["tr"]) for k, v in SPACES.items()}
print(f"hub: {K}-d whitened PCA, mode={HUB_MODE}")
print("   how well each space can REACH the hub (held-out R2):")
for k in SPACES:
    print(f"   {k:11s} {r2(HUB['te'], SPACES[k][te] @ TO_HUB[k]):6.3f}")
print("   This says how well a space can REACH the hub. It does NOT")
print("   predict how well the hub can write back INTO that space -")
print("   measured across seven spaces here, BERT reaches the hub less")
print("   well than GPT-2 yet is three times more writable. What does")
print("   predict writability is the target's own effective rank and")
print("   pair-cosine (geometry table above): ranking vectors inside a")
print("   space whose items are ~+0.999 similar needs precision that")
print("   an extra fitting stage cannot preserve.")

## Test 1 — transitivity: composition through the hub vs a direct fit

In [ ]:
names = list(SPACES)

# TWO metrics, because R2 alone lies about collapsed targets.
#
# R2 = 1 - SSE/SST. A target whose vectors are nearly identical has a
# TINY variance, so the SST denominator is tiny and any error at all
# produces a large negative number. That is a property of the metric,
# not of the reconstruction: this project's own Section C.11 lesson -
# a metric is interpretable only relative to the geometry it is
# computed in - applies to this diagnostic as much as to any result.
#
# R@1 is scale-free: it asks whether the predicted vector points at the
# right item, which no amount of variance compression can distort.
# Report both, and trust R@1 wherever the target's effective rank is
# low (see the geometry table above).
print(f"{'A -> B':26s} {'direct R2':>10s} {'via hub':>9s} {'retained':>9s}"
      f" {'dir R@1':>8s} {'hub R@1':>8s} {'R@1 ret':>8s}")
print("R2 columns are unreliable for collapsed targets; R@1 is not.")
rows = []
for a in names:
    for b in names:
        if a == b: continue
        direct = ridge(SPACES[a][tr], SPACES[b][tr])
        from_hub = ridge(SPACES[a][tr] @ TO_HUB[a], SPACES[b][tr])
        Pd = SPACES[a][te] @ direct
        Ph = (SPACES[a][te] @ TO_HUB[a]) @ from_hub
        rd = r2(SPACES[b][te], Pd)
        rh = r2(SPACES[b][te], Ph)
        kd = recall(l2n(Pd) @ l2n(SPACES[b][te]).T)[1]     # scale-free
        kh = recall(l2n(Ph) @ l2n(SPACES[b][te]).T)[1]
        if rd > 0.05:
            ratio = rh / rd
            rows.append((a, b, rd, rh, ratio, kd, kh))
            # Two ways the ratio becomes meaningless, and they are
            # different problems:
            #   rd small      -> tiny denominator, ratio explodes
            #   rh NEGATIVE   -> the hub path is worse than predicting
            #                    the mean, so there is no "fraction
            #                    retained" to report at all
            if rh < 0:
                shown = "  FAILED *"
            elif rd <= 0.15:
                shown = "   n/a **"
            else:
                shown = f"{100*ratio:8.1f}%"
            kret = (f"{100*kh/kd:7.1f}%" if kd > 0.01 else "      -")
            print(f"{a+' -> '+b:26s} {rd:10.3f} {rh:9.3f} {shown}"
                  f" {kd:8.3f} {kh:8.3f} {kret}")
    #  * the hub cannot reconstruct this target at this width at all
    # ** the direct fit is itself too weak for a ratio to mean anything
    # * ratio omitted where the direct fit itself is near zero (<0.15),
    #   since a tiny denominator makes the percentage uninformative

# Retention depends on how well the TARGET is represented in the hub,
# so report per target - and count the failures separately rather than
# averaging catastrophic negatives into a meaningless number.
print(f"\n{'target space':14s} {'hub R2':>8s} {'mean retention':>15s} "
      f"{'failed':>8s}")
for t in SPACES:
    sub = [r[4] for r in rows if r[1] == t and r[3] >= 0]
    nfail = len([r for r in rows if r[1] == t and r[3] < 0])
    hr = r2(HUB["te"], SPACES[t][te] @ TO_HUB[t])
    val = f"{100*np.mean(sub):14.1f}%" if sub else "             -"
    print(f"{t:14s} {hr:8.3f} {val} {nfail:8d}")
print("\n'failed' counts pairs where the hub path scored a NEGATIVE R2 -")
print("worse than predicting the mean. Those are not low retention, they")
print("are no reconstruction at all, and averaging them in would produce")
print("a number with no meaning. A target with failures and a low hub R2")
print("is simply not carried at this width.")
# average only over targets the hub can actually represent (direct
# fit above 0.15), so one blow-up does not dominate the summary
allr = [r[4] for r in rows if r[2] > 0.15 and r[3] >= 0]
nbad = len([r for r in rows if r[3] < 0])
kr = [r[6]/r[5] for r in rows if r[5] > 0.01]
print(f"\nSCALE-FREE verdict - mean R@1 retention over all {len(kr)} "
      f"scorable pairs: {100*np.mean(kr):.1f}%")
print("This is the number to read when any target has low effective "
      "rank.\n")
print(f"(for reference only) mean R2 retention over the "
      f"{len(allr)} of {len(rows)} pairs R2 could score: "
      f"{100*np.mean(allr):.1f}%")
print("Do NOT read that as the verdict. It is unreliable twice over:")
print("R2 misreports collapsed targets, AND it is averaged over the")
print("subset that did not fail - which excluded the hardest pairs by")
print("construction. Use the scale-free R@1 figure above.\n")

# ---- per-direction breakdown, with broken targets isolated ----
# A source's average is contaminated by any target NOTHING can reach:
# every encoder scores badly writing into a collapsed space, which says
# nothing about that encoder. So identify structurally-broken targets
# first, then report source averages both with and without them.
GATE = 0.70
broken = [t for t in names
          if len([r for r in rows if r[1] == t and r[5] > 0.01]) and
          np.mean([r[6]/r[5] for r in rows if r[1] == t and r[5] > 0.01])
          < 0.50]
if broken:
    print(f"structurally-broken TARGETS (nothing writes into them): "
          f"{', '.join(broken)}")
    print("excluded from the source averages below, and reported "
          "separately\n")

print(f"{'space':12s} {'as SOURCE':>10s} {'as TARGET':>10s}"
      f" {'needs map':>10s} {'-> bge?':>9s}   reading")
for k in names:
    src = [r[6]/r[5] for r in rows
           if r[0] == k and r[5] > 0.01 and r[1] not in broken]
    tgt = [r[6]/r[5] for r in rows if r[1] == k and r[5] > 0.01]
    if not src or not tgt:
        continue
    ms, mt = 100*np.mean(src), 100*np.mean(tgt)
    # can this space be written INTO bge's space through the hub?
    to_bge = [r[6]/r[5] for r in rows
              if r[0] == k and r[1] == "txt_bge" and r[5] > 0.01]
    bge = f"{100*to_bge[0]:8.1f}%" if to_bge else "        -"
    note = ("carried both ways" if mt >= 100*GATE else
            "READ-ONLY - readable out of, not writable into")
    print(f"{k:12s} {ms:9.1f}% {mt:9.1f}% {'yes':>10s} {bge:>9s}   {note}")

allk = [r[6]/r[5] for r in rows if r[5] > 0.01]
okk = [r[6]/r[5] for r in rows if r[5] > 0.01 and r[1] not in broken]
print(f"\nmean R@1 retention, all {len(allk)} pairs        : "
      f"{100*np.mean(allk):.1f}%")
print(f"mean R@1 retention, excluding broken targets: "
      f"{100*np.mean(okk):.1f}%   ({len(okk)} pairs, "
      f"range {100*min(okk):.1f}-{100*max(okk):.1f}%)")
print()
print("EVERY space needs its own linear map into the hub - that is the")
print("design: N encoders, N maps, one shared basis. What differs is")
print("whether the hub can write BACK into a space, shown above. The")
print("'-> bge?' column is the one that matters for the caption head,")
print("which reads out into bge-m3 space: a space that reaches bge well")
print("can drive the head; one that does not can still be a source for")
print("everything else.")
print()
print("A space scoring high as a SOURCE and low as a TARGET is not badly")
print("mapped, and the hub has not lost it - the hub reconstructs such a")
print("space essentially perfectly. The cause is how similar the TARGET's")
print("own items are to each other. Retrieval INTO a space means ranking")
print("that space's vectors against one another, and in a collapsed space")
print("every item sits at ~+0.999 cosine from every other, so the")
print("distinctions live in the last decimal place. The hub path adds one")
print("extra fitting stage through a bottleneck, and that error is larger")
print("than the gaps it must preserve. In a healthy target, items sit far")
print("enough apart that the same error is harmless - and the hub's")
print("denoising can even help, which is why some pairs exceed 100%.")
print("Compare each space's pair-cosine in the geometry table above: it")
print("predicts this column better than anything else does.")

## Choosing HUB_DIM by measurement

`HUB_DIM` trades two things against each other. Too narrow and the hub
cannot RECONSTRUCT a wide target: on the Experiment B spaces, retention
into the widest target ran at -444 per cent with a 256-d hub and +84 per
cent at 2048. Too wide and the maps into the hub become underpowered -
rows per hub dimension falls, and this project's own threshold is five.

The sweep below reports both, plus the transfer result, so the choice is
made from evidence rather than argued. The singular value decomposition
is computed once and sliced, so additional widths are nearly free.

Verification AUC is included alongside retrieval. It asks the weaker
question of Appendix D.2 - can a threshold separate a true pair from a
random one - and a transferred component may hold up far better there
than its R@1 suggests. That is supplementary evidence, not the headline.

In [ ]:
# ---- HUB_DIM x ALPHA sweep, transfer-first ----
# The transfer test wants a NARROW hub (whitened PCA amplifies tail
# noise directions, which dominate cosine geometry past a threshold);
# reconstruction wants a WIDE one. They are not reconcilable, and that
# is the finding. Alpha shifts the cliff slightly but does not move it
# past the narrow setting, so this grid exists to LOCATE the cliff on
# your data, not to defeat it.
if HUB_MODE == "single":
    _ref = SPACES[HUB_REF][tr]
else:
    _ref = np.hstack([(SPACES[k][tr] - SPACES[k][tr].mean(0)) /
                      (SPACES[k][tr].std(0).mean() + 1e-12)
                      for k in SPACES])
_mu = _ref.mean(0)
_U, _sv, _VT = np.linalg.svd(_ref - _mu, full_matrices=False)

def _co(which, K):
    B = _VT[:K].T / (_sv[:K] / np.sqrt(len(_ref)))
    if HUB_MODE == "single":
        return (SPACES[HUB_REF][which] - _mu) @ B
    X = np.hstack([(SPACES[k][which] - SPACES[k][tr].mean(0)) /
                   (SPACES[k][tr].std(0).mean() + 1e-12) for k in SPACES])
    return (X - _mu) @ B

def _auc(pos, neg):
    y = np.r_[np.ones(len(pos)), np.zeros(len(neg))]; s = np.r_[pos, neg]
    o = np.argsort(-s); y = y[o]
    return float(np.trapezoid(np.cumsum(y)/y.sum(),
                              np.cumsum(1-y)/(1-y).sum()))

_don = [k for k in SPACES if k.startswith("img_")]
assert "txt_bge" in SPACES and len(_don) > 1, "need txt_bge + 2 image spaces"
_gal = l2n(SPACES["txt_bge"][te])
_rng = np.random.default_rng(1)
_j = _rng.permutation(len(te))
_j = np.where(_j == np.arange(len(te)), (_j + 1) % len(te), _j)

DIMS   = [k for k in (128, 256, 512, 768, 1024) if k <= len(_sv)]
ALPHAS = [1e-2, 1e-1, 1.0, 10.0]

print("TRANSFER R@1  (head trained on", _don[0], "-> unseen", _don[1] + ")")
print(f"{'HUB_DIM':>8s} {'rows/dim':>9s}" +
      "".join(f"{'a='+str(a):>9s}" for a in ALPHAS))
best = (-1, None, None)
grid = {}
for K in DIMS:
    htr, hte = _co(tr, K), _co(te, K)
    line = f"{K:8d} {len(tr)/K:9.1f}"
    for a in ALPHAS:
        TH = {k: np.linalg.solve(SPACES[k][tr].T @ SPACES[k][tr] +
                                 a*np.eye(SPACES[k].shape[1]),
                                 SPACES[k][tr].T @ htr) for k in SPACES}
        hd = np.linalg.solve((SPACES[_don[0]][tr]@TH[_don[0]]).T @
                             (SPACES[_don[0]][tr]@TH[_don[0]]) +
                             a*np.eye(K),
                             (SPACES[_don[0]][tr]@TH[_don[0]]).T @
                             SPACES["txt_bge"][tr])
        P = l2n((SPACES[_don[1]][te] @ TH[_don[1]]) @ hd)
        r1 = recall(P @ _gal.T)[1]
        grid[(K, a)] = (r1, _auc((P*_gal).sum(1), (P*_gal[_j]).sum(1)))
        line += f"{r1:9.3f}"
        if r1 > best[0] and len(tr)/K >= 5:      # respect the rows/dim floor
            best = (r1, K, a)
    print(line)
print(f"\nbest transfer with rows/dim >= 5: "
      f"R@1 {best[0]:.3f} at HUB_DIM={best[1]}, ALPHA={best[2]} "
      f"(verif AUC {grid[(best[1],best[2])][1]:.3f})")
print("set HUB_DIM and ALPHA in the config cell to these, then re-run the")
print("transfer cell for the headline numbers.")

## Test 2 — zero-shot transfer of a trained component

The head maps hub vectors to the bge-m3 caption space and is trained on
**one** encoder's hub vectors only. It is then applied to other encoders'
hub vectors and scored by caption retrieval on the same held-out rows.
Two controls: the native head fitted directly on each encoder (an upper
reference), and the same head fed through a random map (which must sit
at chance, or the test proves nothing).

In [ ]:
assert "txt_bge" in SPACES, "needs the bge-m3 space - run E1 first"
TARGET = "txt_bge"
donors = [k for k in SPACES if k.startswith("img_")]
TRAIN_ON = donors[0]                      # smallest encoder available

head = ridge(SPACES[TRAIN_ON][tr] @ TO_HUB[TRAIN_ON],
             SPACES[TARGET][tr])
gal = l2n(SPACES[TARGET][te])
print(f"head trained ONLY on {TRAIN_ON}, applied through the hub\n")
print(f"{'encoder':12s} {'R@1':>7s} {'R@5':>7s} {'R@10':>7s} "
      f"{'vs native R@1':>14s}")
for enc in donors:
    P = l2n((SPACES[enc][te] @ TO_HUB[enc]) @ head)
    r = recall(P @ gal.T)
    nat = ridge(SPACES[enc][tr], SPACES[TARGET][tr])
    rn = recall(l2n(SPACES[enc][te] @ nat) @ gal.T)
    tag = "  <- trained here" if enc == TRAIN_ON else "  <- NEVER SEEN"
    print(f"{enc:12s} {r[1]:7.3f} {r[5]:7.3f} {r[10]:7.3f} "
          f"{100*r[1]/max(rn[1],1e-9):13.1f}%{tag}")

Rrand = rng.standard_normal(TO_HUB[donors[-1]].shape) / \
        np.sqrt(SPACES[donors[-1]].shape[1])
rr = recall(l2n((SPACES[donors[-1]][te] @ Rrand) @ head) @ gal.T)
print(f"\ncontrol - random map into the hub: R@1 {rr[1]:.3f} "
      f"(chance {1/N_EVAL:.3f})")

## Seeing the shared space

Everything above measures the hub. This draws it. Three views, all from
the hub coordinates this run produced:

**A — the space itself.** A 2D projection of hub coordinates. Each item
is encoded by every registered encoder, giving one point per encoder,
and the points belonging to the same item are joined. If the hub works,
those clusters are tight and separate from one another — the same
picture the agreement matrix reports as numbers.

**B — the spectra.** Every space's variance profile on one axis. A
collapsed space is visible immediately: its curve falls off a cliff
while healthy ones decay gently. This is the effective-rank column,
drawn.

**C — direction agreement.** For one item, the cosine between every
pair of encoders' hub vectors, against the same quantity for two
different items. The gap between the two is what makes retrieval
possible at all.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mp

def plot_hub(n_items=12, seed=0):
    """Draw the hub: the space, the spectra, and the direction gap."""
    r = np.random.default_rng(seed)
    H = {k: SPACES[k][te] @ TO_HUB[k] for k in names}
    pick = r.permutation(len(te))[:n_items]

    fig = plt.figure(figsize=(13.6, 4.6))
    gs = fig.add_gridspec(1, 3, width_ratios=[1.15, 1, 0.95], wspace=0.30)
    cols = plt.cm.tab10(np.linspace(0, 1, max(len(names), 3)))
    NAVY = "#1a1a2e"

    # --- A: the hub, projected ---
    ax = fig.add_subplot(gs[0])
    allpts = np.vstack([H[k][pick] for k in names])
    C = allpts - allpts.mean(0)
    _, _, Vt = np.linalg.svd(C, full_matrices=False)
    P = {k: (H[k][pick] - allpts.mean(0)) @ Vt[:2].T for k in names}
    for i in range(n_items):                     # join the same item
        xs = [P[k][i, 0] for k in names]; ys = [P[k][i, 1] for k in names]
        cx, cy = float(np.mean(xs)), float(np.mean(ys))
        for x, y in zip(xs, ys):
            ax.plot([cx, x], [cy, y], c="#95a5a6", lw=0.7, alpha=0.75,
                    zorder=1)
    for ci, k in enumerate(names):
        ax.scatter(*P[k].T, s=26, color=cols[ci], label=k, zorder=3,
                   edgecolors="white", linewidths=0.4)
    ax.set_xticks([]); ax.set_yticks([])
    ax.set_title(f"A · the hub itself\n{n_items} items x {len(names)} "
                 f"encoders; lines join ONE item", fontsize=9.8,
                 color=NAVY)
    ax.legend(fontsize=6.2, frameon=False, ncol=2, loc="upper center",
              bbox_to_anchor=(0.5, -0.02))

    # --- B: spectra ---
    ax = fig.add_subplot(gs[1])
    for ci, k in enumerate(names):
        Z = SPACES[k][tr] - SPACES[k][tr].mean(0)
        s = np.linalg.svd(Z, full_matrices=False, compute_uv=False)
        share = s**2 / (s**2).sum()
        er = float(np.exp(-(share[share > 0] *
                            np.log(share[share > 0])).sum()))
        ax.plot(share[:120], lw=1.5, color=cols[ci],
                label=f"{k} (er {er:.0f})")
    ax.set_yscale("log")
    ax.set_xlabel("principal direction", fontsize=8.2)
    ax.set_ylabel("share of variance (log)", fontsize=8.2)
    ax.set_title("B · every space's spectrum\na cliff = a collapsed space",
                 fontsize=9.8, color=NAVY)
    ax.legend(fontsize=5.8, frameon=False); ax.tick_params(labelsize=7)

    # --- C: direction agreement ---
    ax = fig.add_subplot(gs[2])
    same, diff = [], []
    for i in r.permutation(len(te))[:200]:
        v = np.stack([l2n(H[k][i:i+1])[0] for k in names])
        j = int(r.integers(len(te)))
        w = np.stack([l2n(H[k][j:j+1])[0] for k in names])
        for a in range(len(names)):
            for b in range(a+1, len(names)):
                same.append(float(v[a] @ v[b]))
                diff.append(float(v[a] @ w[b]))
    bins = np.linspace(-1, 1, 50)
    ax.hist(diff, bins=bins, color="#c0392b", alpha=0.55,
            label="different items")
    ax.hist(same, bins=bins, color="#0f766e", alpha=0.62,
            label="SAME item, two encoders")
    ax.axvline(float(np.mean(same)), c="#0f766e", ls="--", lw=1.2)
    ax.axvline(float(np.mean(diff)), c="#c0392b", ls="--", lw=1.2)
    ax.set_xlabel("cosine between hub vectors", fontsize=8.2)
    ax.set_title(f"C · the gap that makes retrieval work\n"
                 f"same {np.mean(same):+.2f} vs different "
                 f"{np.mean(diff):+.2f}", fontsize=9.8, color=NAVY)
    ax.legend(fontsize=7, frameon=False); ax.tick_params(labelsize=7)

    for x in fig.axes:
        x.spines["top"].set_visible(False)
        x.spines["right"].set_visible(False)
    fig.suptitle("The shared space, drawn from this run's own hub "
                 "coordinates", fontsize=11, color=NAVY)
    fig.subplots_adjust(left=0.06, right=0.97, top=0.80, bottom=0.22,
                        wspace=0.30)
    plt.show()

plot_hub()

## How to read this

Transitivity says the hub is not an arbitrary waypoint: if composing
A→hub→B retains most of a direct A→B fit, the encoders are being
described in one coordinate system rather than by a web of bilateral
agreements.

Zero-shot transfer is the constructive claim. A head trained on one
encoder's hub vectors, applied to a different encoder that it never saw
in any form, is a working system assembled from parts that were never
trained together — the shared space demonstrated by use.

Report the retention percentages, not just whether it "works", and
report the random-map control alongside: without it, a transfer number
means nothing. Note also the scope — the encoders here come from two
families on one image domain, so this exhibits a shared space for these
five, not for all models.